# 10 Proxy | _Kamil Bartocha_ | wersja 2.0

## Rozklad jazdy

1. ❓ Problem: kontrola dostepu do obiektu
2. 🔄 Typy proxy (wirtualne, ochronne, cache'ujace)
3. 🔮 Implementacja z `__getattr__`
4. 🔗 `weakref.proxy`
5. 🆚 Proxy vs Decorator

## 1. 🔹 Problem: kontrola dostepu do obiektu

Proxy (Pelnomocnik) to wzorzec strukturalny dostarczajacy
obiekt zastepczy ktory kontroluje dostep do innego obiektu.

Analogacja: karta kredytowa - pelnomocnik dla Twojego konta bankowego.
Moze sprawdzac limit, logowac transakcje, blokowac w nocy.

Kiedy uzywamy Proxy:
- Lazy loading: obiekt drogi - tworzymy go tylko gdy potrzebny
- Kontrola dostepu: sprawdzamy uprawnienia przed wywolaniem
- Cache: zapamiętujemy wyniki kosztownych operacji
- Logowanie: sledzeznie wywolan metod
- Zdalne proxy: obiekt jest na innym serwerze

Kluczowa rola: Proxy i oryginal implementuja ten sam interfejs.
Klient nie wie czy ma do czynienia z prawdziwym obiektem czy proxy.

> 💡 Proxy to NIE to samo co Decorator (GoF). Proxy
> kontroluje dostep, Decorator dodaje zachowanie. Ale w Pythonie
> granica jest plynna - oba uzyja __getattr__.

In [ ]:
import time

# Obiekt drogi w tworzeniu
class HeavyReport:
    def __init__(self, name: str):
        self.name = name
        print(f'[HeavyReport] Generating {name}...')  # kosztowne
        time.sleep(0.01)
        self._data = list(range(10_000))

    def show(self) -> str:
        return f'Report({self.name}, {len(self._data)} rows)'

# BEZ proxy: zawsze tworzymy natychmiast
print('Creating report...')
report = HeavyReport('annual_sales')
print(report.show())  # uzywamy natychmiast - OK

print()

# Problem: jesli uzytkownik nigdy nie otworzy raportu - tracilimy czas
print('Creating 5 reports (moze nigdy nie otworzone):')
reports = [HeavyReport(f'report_{i}') for i in range(5)]
print('Created all 5 - ale moze 4 z nich nigdy nie uzyte!')

---

### 🐍 Cwiczenia - problem

1. Zmierz czas tworzenia 10 obiektow `HeavyReport` vs 10 obiektow
   `LazyReportProxy` (bez wywolania `show()`).
2. Napisz liste 3 scenariuszy gdzie lazy loading przynosi realne
   korzysci wydajnosciowe.
3. *(Trudniejsze)* Zmierz zuzycie pamieci dla listy 100
   `HeavyReport` vs 100 proxy (uzywajac `sys.getsizeof`).

In [ ]:
# Cwiczenie 1: pomiar czasu
class LazyReportProxy:
    def __init__(self, name: str):
        self._name = name
        self._report = None
    def show(self) -> str:
        if self._report is None:
            self._report = HeavyReport(self._name)
        return self._report.show()

start = time.perf_counter()
heavy = [HeavyReport(f'r{i}') for i in range(10)]
t_heavy = time.perf_counter() - start

start = time.perf_counter()
proxies = [LazyReportProxy(f'r{i}') for i in range(10)]
t_proxy = time.perf_counter() - start

print(f'HeavyReport x10: {t_heavy:.4f}s')
print(f'LazyProxy x10:   {t_proxy:.6f}s')
print(f'Szybszy: {t_heavy/t_proxy:.0f}x')

In [ ]:
# Cwiczenie 2: scenariusze lazy loading
scenarios = [
    'Miniaturki zdjec w galerii - laduj tylko widoczne',
    'Polaczenia do bazy danych - otworz dopiero przy pierwszym zapytaniu',
    'Pliki konfiguracyjne - parsuj dopiero gdy konfiguracja potrzebna',
]
for i, s in enumerate(scenarios, 1):
    print(f'{i}. {s}')

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: zuzycie pamieci
import sys

heavy_list = [HeavyReport(f'r{i}') for i in range(5)]   # male dla testu
proxy_list = [LazyReportProxy(f'r{i}') for i in range(100)]

heavy_size = sum(sys.getsizeof(r._data) for r in heavy_list)
proxy_size = sum(sys.getsizeof(p) for p in proxy_list)

print(f'5 HeavyReport (data): {heavy_size:,} bytes')
print(f'100 LazyProxy: {proxy_size:,} bytes')
print('Proxy nie laduje danych dopoki nie wywolamy show()')

## 2. 🔹 Typy proxy (wirtualne, ochronne, cache'ujace)

Trzy glowne typy proxy:

**Proxy wirtualne (Virtual Proxy)**:
- Odkłada tworzenie drogiego obiektu do pierwszego uzycia
- Wzorzec lazy loading
- Przyklad: miniaturki zdjec, polaczenia do bazy

**Proxy ochronne (Protection Proxy)**:
- Sprawdza uprawnienia przed udostepnieniem metod
- Ten sam interfejs co oryginal
- Przyklad: systemy kontroli dostepu, role-based access

**Proxy cache'ujace (Caching Proxy)**:
- Zapamiętuje wyniki kosztownych operacji
- Zwraca cached wynik zamiast wywolywac oryginalna operacje
- Przyklad: wyniki zapytan do bazy, odpowiedzi API

**Proxy logujace (Logging Proxy)**:
- Sledzi wszystkie wywolania metod
- Uzywane do debugowania i audytu

**Zdalne proxy (Remote Proxy)**:
- Reprezentuje obiekt na zdalnym serwerze
- Przyklad: XML-RPC, gRPC stubs

In [ ]:
from abc import ABC, abstractmethod
import time

class Image(ABC):
    @abstractmethod
    def display(self) -> None: ...

class RealImage(Image):
    def __init__(self, filename: str):
        self.filename = filename
        print(f'Loading: {filename}')
        time.sleep(0.01)
    def display(self) -> None:
        print(f'Displaying: {self.filename}')

# PROXY WIRTUALNE
class LazyImageProxy(Image):
    def __init__(self, filename: str):
        self.filename = filename
        self._real: RealImage | None = None
    def display(self) -> None:
        if self._real is None:
            self._real = RealImage(self.filename)
        self._real.display()

# PROXY OCHRONNE
class DatabaseService:
    def read(self, sql: str) -> list: return [{'id': 1}]
    def write(self, sql: str) -> None: print(f'Write: {sql}')
    def delete(self, sql: str) -> None: print(f'Delete: {sql}')

class ProtectionProxy(DatabaseService):
    def __init__(self, service: DatabaseService, role: str):
        self._service = service
        self._role = role
    def read(self, sql: str) -> list:
        return self._service.read(sql)  # wszyscy moga czytac
    def write(self, sql: str) -> None:
        if self._role not in ('editor', 'admin'):
            raise PermissionError(f'Role {self._role!r} cannot write')
        self._service.write(sql)
    def delete(self, sql: str) -> None:
        if self._role != 'admin':
            raise PermissionError(f'Role {self._role!r} cannot delete')
        self._service.delete(sql)

# PROXY CACHE'UJACE
class WeatherAPI:
    def get_weather(self, city: str) -> dict:
        print(f'[API] Fetching weather for {city}')
        time.sleep(0.01)
        return {'city': city, 'temp': 20}

class CachingWeatherProxy:
    TTL = 60
    def __init__(self, api: WeatherAPI):
        self._api = api
        self._cache: dict[str, tuple] = {}
    def get_weather(self, city: str) -> dict:
        now = time.time()
        if city in self._cache:
            data, ts = self._cache[city]
            if now - ts < self.TTL:
                print(f'[Cache] {city}')
                return data
        result = self._api.get_weather(city)
        self._cache[city] = (result, now)
        return result

print('--- Virtual Proxy ---')
proxy = LazyImageProxy('photo.jpg')
print('Created (not loaded yet)')
proxy.display()  # laduje teraz
proxy.display()  # juz zaladowane

print('--- Protection Proxy ---')
viewer = ProtectionProxy(DatabaseService(), 'viewer')
print(viewer.read('SELECT *'))
try:
    viewer.write('INSERT ...')
except PermissionError as e:
    print(f'Denied: {e}')

print('--- Caching Proxy ---')
weather = CachingWeatherProxy(WeatherAPI())
print(weather.get_weather('Warsaw'))
print(weather.get_weather('Warsaw'))  # z cache

---

### 🐍 Cwiczenia - typy proxy

1. Napisz `LazyReportProxy` dla `LargeReport.generate()` -
   generowanie tylko przy pierwszym wywolaniu.
2. Napisz `AccessControlProxy(service, role)` dla `UserService`
   - admin moze wszystko, viewer tylko `get_users()`.
3. *(Trudniejsze)* Napisz `CachedWeatherProxy` z TTL w sekundach.
   Przetestuj ze sztuczna zmiana czasu (`time.time` monkeypatching).

In [ ]:
# Cwiczenie 1: LazyReportProxy
class LargeReport:
    def __init__(self, name: str):
        self.name = name
        print(f'Generating report: {name}...')
        self._data = list(range(1000))
    def generate(self) -> str:
        return f'Report({self.name}, {len(self._data)} rows)'

class LazyReportProxy:
    def __init__(self, name: str):
        self._name = name
        self._report = None
    def generate(self) -> str:
        ...

proxy = LazyReportProxy('annual_sales')
print('Proxy created (no generation yet)')
print(proxy.generate())  # generuje teraz
print(proxy.generate())  # bez ponownego generowania

In [ ]:
# Cwiczenie 2: AccessControlProxy
class UserService:
    def get_users(self) -> list: return [{'id': 1, 'name': 'Alice'}]
    def create_user(self, name: str) -> dict:
        print(f'Creating: {name}')
        return {'id': 2, 'name': name}
    def delete_user(self, uid: int) -> None:
        print(f'Deleting: {uid}')

class AccessControlProxy:
    ALLOWED = {
        'admin': {'get_users', 'create_user', 'delete_user'},
        'viewer': {'get_users'},
    }
    def __init__(self, service: UserService, role: str):
        ...
    def get_users(self) -> list: ...
    def create_user(self, name: str) -> dict: ...
    def delete_user(self, uid: int) -> None: ...

admin = AccessControlProxy(UserService(), 'admin')
viewer = AccessControlProxy(UserService(), 'viewer')
print(admin.get_users())
admin.create_user('Bob')
print(viewer.get_users())
try:
    viewer.create_user('Eve')
except PermissionError as e:
    print(f'Permission: {e}')

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: CachedWeatherProxy z TTL
class CachedWeatherProxy:
    def __init__(self, api: WeatherAPI, ttl: float = 60):
        self._api = api
        self._ttl = ttl
        self._cache: dict[str, tuple] = {}

    def get_weather(self, city: str) -> dict:
        # hint: cache jako {city: (result, timestamp)}
        ...

# Test z krotkiim TTL
short_cache = CachedWeatherProxy(WeatherAPI(), ttl=0.05)
print(short_cache.get_weather('Warsaw'))  # API call
print(short_cache.get_weather('Warsaw'))  # cache hit
time.sleep(0.1)  # TTL wygaslo
print(short_cache.get_weather('Warsaw'))  # API call ponownie

## 3. 🔹 Implementacja z `__getattr__`

`__getattr__` pozwala na dynamiczne proxy bez koniecznosci
recznego implementowania kazdej metody.

Kluczowe metody specjalne:
- `__getattr__(name)` - wywolane gdy atrybut NIE znaleziony
- `__getattribute__(name)` - wywolane ZAWSZE przy dostepu
- `__setattr__(name, value)` - przechwytuje przypisania

Trick z `object.__setattr__`:
Gdy implementujemy `__setattr__`, musimy uzywac
`object.__setattr__(self, name, value)` dla atrybutow
samego proxy - inaczej tworzymy nieskonczona rekurencje.

Ten sam trick dotyczy `__getattribute__`:
`object.__getattribute__(self, '_target')` pomija nasz override.

In [ ]:
# LoggingProxy przez __getattr__
class LoggingProxy:
    def __init__(self, target):
        # uzyj object.__setattr__ aby ominac nasz __setattr__
        object.__setattr__(self, '_target', target)

    def __getattr__(self, name: str):
        attr = getattr(object.__getattribute__(self, '_target'), name)
        if callable(attr):
            def logged(*args, **kwargs):
                print(f'CALL: {name}({args}, {kwargs})')
                result = attr(*args, **kwargs)
                print(f'RETURN: {name} -> {result!r}')
                return result
            return logged
        return attr

class Calculator:
    def add(self, a: int, b: int) -> int: return a + b
    def multiply(self, a: int, b: int) -> int: return a * b
    def version(self) -> str: return '1.0'

calc = LoggingProxy(Calculator())
calc.add(3, 4)
calc.multiply(5, 6)
print(calc.version())  # rowniez przez proxy


# Proxy z filtrowaniem metod
class SelectiveProxy:
    """Przekazuje tylko niektorych metody."""
    def __init__(self, target, allowed: set):
        object.__setattr__(self, '_target', target)
        object.__setattr__(self, '_allowed', allowed)

    def __getattr__(self, name: str):
        if name not in object.__getattribute__(self, '_allowed'):
            raise AttributeError(f'Method {name!r} not allowed through proxy')
        return getattr(object.__getattribute__(self, '_target'), name)

class FileService:
    def read(self, path: str) -> str: return f'content:{path}'
    def write(self, path: str, data: str) -> None: print(f'Written:{path}')
    def delete(self, path: str) -> None: print(f'Deleted:{path}')

read_only = SelectiveProxy(FileService(), {'read'})
print(read_only.read('/etc/hosts'))  # dziala
try:
    read_only.write('/etc/hosts', 'hack')  # blokowane
except AttributeError as e:
    print(f'Blocked: {e}')

---

### 🐍 Cwiczenia - `__getattr__`

1. Napisz `LoggingProxy(target)` logujacy kazde wywolanie metody
   w formacie `CALL: method_name(args) -> result`.
2. Napisz `TimingProxy(target)` mierzacy czas kazdego wywolania.
3. *(Trudniejsze)* Napisz `ValidatingProxy(target, validators: dict)`
   gdzie `validators` to slownik `{method_name: callable}` sprawdzajacy
   argumenty przed przekazaniem do metody.

In [ ]:
# Cwiczenie 1: LoggingProxy przez __getattr__
class LoggingProxy:
    def __init__(self, target):
        object.__setattr__(self, '_target', target)

    def __getattr__(self, name: str):
        ...

class StringProcessor:
    def upper(self, text: str) -> str: return text.upper()
    def reverse(self, text: str) -> str: return text[::-1]
    def word_count(self, text: str) -> int: return len(text.split())

proc = LoggingProxy(StringProcessor())
proc.upper('hello world')
proc.reverse('Python')
proc.word_count('design patterns are cool')

In [ ]:
# Cwiczenie 2: TimingProxy
class TimingProxy:
    def __init__(self, target):
        object.__setattr__(self, '_target', target)

    def __getattr__(self, name: str):
        ...

class DataProcessor:
    def sort(self, data: list) -> list: return sorted(data)
    def unique(self, data: list) -> list: return list(set(data))
    def stats(self, data: list) -> dict:
        return {'min': min(data), 'max': max(data), 'mean': sum(data)/len(data)}

import random
data = [random.randint(1, 100) for _ in range(10_000)]
timed = TimingProxy(DataProcessor())
timed.sort(data)
timed.unique(data)
timed.stats(data)

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: ValidatingProxy
class ValidatingProxy:
    def __init__(self, target, validators: dict):
        # hint: validators = {'method_name': lambda *args: True/raise}
        object.__setattr__(self, '_target', target)
        object.__setattr__(self, '_validators', validators)

    def __getattr__(self, name: str):
        ...

class BankAccount:
    def __init__(self, balance: float): self.balance = balance
    def deposit(self, amount: float) -> float:
        self.balance += amount; return self.balance
    def withdraw(self, amount: float) -> float:
        self.balance -= amount; return self.balance

def validate_positive(amount, *args):
    if amount <= 0: raise ValueError(f'Amount must be positive, got {amount}')

def validate_sufficient(amount, *args, account=None):
    pass  # potrzebowaloby dostepu do obiektu

account = ValidatingProxy(
    BankAccount(100.0),
    validators={'deposit': validate_positive, 'withdraw': validate_positive}
)
print(account.deposit(50))
try:
    account.deposit(-10)
except ValueError as e:
    print(f'Validation: {e}')

## 4. 🔹 `weakref.proxy`

`weakref.proxy(obj)` tworzy proxy ze slaba referencja (weak reference):
- Slaba referencja NIE zapobiega usunięciu obiektu przez GC
- Gdy oryginalny obiekt zostanie usuniety, proxy rzuca `ReferenceError`
- Uzywamy gdy chcemy proxy ktore nie przedluza zycia obiektu

Silna vs slaba referencja:
- Silna: `x = obj` - GC nie usunie obiektu dopoki x istnieje
- Slaba: `x = weakref.ref(obj)` - GC moze usunac obiekt
- Proxy: `x = weakref.proxy(obj)` - jak slaba ale przezroczyste API

Zastosowania:
- Cache: nie chcemy trzymac obiektow w pamieci sztucznie
- Obserwatorzy: Observer nie powinien blokowac usuniecia Subject
- Unikanie cyklow referencji (memory leaks)

> 💡 `weakref.proxy` jest gotowym proxy wbudowanym w Python.
> Dziala przezroczyscie - wyglada jak oryginalny obiekt.

In [ ]:
import weakref
import gc

class Resource:
    def __init__(self, name: str):
        self.name = name
        print(f'Resource created: {name}')
    def __del__(self):
        print(f'Resource destroyed: {self.name}')
    def use(self) -> str:
        return f'Using {self.name}'

# Slaba referencja przez weakref.ref
obj = Resource('database_connection')
weak_ref = weakref.ref(obj)
print('Oryginal:', obj.use())
print('Przez ref:', weak_ref().use())  # wywolujemy weak_ref() aby dostac obiekt

# proxy - bardziej przezroczyste
proxy = weakref.proxy(obj)
print('Przez proxy:', proxy.use())  # bez dodatkowego ()!
print('Proxy name:', proxy.name)

# Co sie stanie gdy oryginal zostanie usuniety?
print('Usuwamy oryginal...')
del obj
gc.collect()

try:
    print(proxy.use())  # ReferenceError!
except ReferenceError as e:
    print(f'Proxy niedostepne: {e}')


# weakref do cache - nie zapobiega GC
class ImageCache:
    def __init__(self):
        self._cache: dict[str, weakref.ref] = {}

    def get(self, key: str):
        ref = self._cache.get(key)
        if ref is not None:
            obj = ref()  # wywolaj aby sprawdzic czy istnieje
            if obj is not None:
                print(f'Cache hit: {key}')
                return obj
        return None

    def put(self, key: str, obj) -> None:
        self._cache[key] = weakref.ref(obj)

cache = ImageCache()
img = Resource('profile_photo')
cache.put('user_1', img)
print(cache.get('user_1').use())
del img
gc.collect()
print('After GC:', cache.get('user_1'))  # None - obiekt usuniety

---

### 🐍 Cwiczenia - weakref

1. Sprawdz czy `weakref.proxy` zachowuje sie jak oryginalny obiekt
   dla operacji: `==`, `str()`, `len()`, `in` operator.
2. Napisz `WeakCache` przechowujacy wartosci jako slabe referencje.
   Sprawdz ze cache automatycznie czysci sie po `del`.
3. *(Trudniejsze)* Zaimplementuj wzorzec Observer gdzie Subject
   trzyma observerow jako slabe referencje. Sprawdz ze usuniety
   observer jest automatycznie wyrejestrowywany.

In [ ]:
# Cwiczenie 1: weakref.proxy - operacje
class DataList:
    def __init__(self, data: list):
        self.data = data
    def __len__(self): return len(self.data)
    def __eq__(self, other): return self.data == other.data
    def __contains__(self, item): return item in self.data
    def __str__(self): return f'DataList({self.data})'

original = DataList([1, 2, 3, 4, 5])
proxy = weakref.proxy(original)

print('len:', len(proxy))           # dziala jak original?
print('str:', str(proxy))           # dziala jak original?
print('in:', 3 in proxy)            # dziala jak original?
print('eq:', proxy == original)     # dziala jak original?

In [ ]:
# Cwiczenie 2: WeakCache
class WeakCache:
    def __init__(self):
        self._cache: dict[str, weakref.ref] = {}

    def put(self, key: str, value) -> None:
        self._cache[key] = weakref.ref(value)

    def get(self, key: str):
        ref = self._cache.get(key)
        if ref is not None:
            return ref()
        return None

    def size(self) -> int:
        return sum(1 for ref in self._cache.values() if ref() is not None)

cache = WeakCache()
obj1 = Resource('config')
obj2 = Resource('session')
cache.put('config', obj1)
cache.put('session', obj2)
print(f'Cache size: {cache.size()}')
del obj1
gc.collect()
print(f'Cache size after del: {cache.size()}')
print(f'config: {cache.get("config")}')
print(f'session: {cache.get("session").use()}')

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: Observer z weak references
class WeakObserverMixin:
    """Subject trzyma observerow jako slabe referencje."""
    def __init__(self):
        self._observers: list[weakref.ref] = []

    def subscribe(self, observer) -> None:
        # hint: uzyj weakref.ref z finalizerem do auto-usuwania
        self._observers.append(weakref.ref(observer))

    def notify(self, event: str, data: dict) -> None:
        alive = []
        for ref in self._observers:
            obs = ref()
            if obs is not None:
                obs.update(event, data)
                alive.append(ref)
        self._observers = alive  # usun martwe referencje

class StockMarket(WeakObserverMixin):
    def __init__(self):
        super().__init__()
        self.price = 100
    def set_price(self, price: float) -> None:
        self.price = price
        self.notify('price_changed', {'price': price})

class StockAlert:
    def __init__(self, name: str): self.name = name
    def update(self, event: str, data: dict) -> None:
        print(f'[{self.name}] {event}: {data}')

market = StockMarket()
alert1 = StockAlert('Alert1')
alert2 = StockAlert('Alert2')
market.subscribe(alert1)
market.subscribe(alert2)
market.set_price(105)
print('Usuwamy alert1...')
del alert1
gc.collect()
market.set_price(110)  # tylko alert2 powinien dostac

## 5. 🔹 Proxy vs Decorator

Proxy i Decorator wyglada podobnie w kodzie - oba 'owijaja'
object i deleguja do niego. Rozroznienie jest w intencji:

| Kryterium | Proxy | Decorator |
|---|---|---|
| Cel | Kontrola dostepu / lifecycle | Dodanie zachowania |
| Interfejs | Ten sam | Ten sam |
| Znajomosc obiektu | Proxy tworzy lub otrzymuje | Decorator zawsze dostaje |
| Skladanie | Zazwyczaj jedno proxy | Wiele dekoratorow na raz |
| Zycie obiektu | Proxy moze kontrolowac (lazy, cache) | Decorator nie zmienia |

W praktyce Pythona granica jest plynna:
- `@functools.lru_cache` jest i Proxy (cache) i Decorator (@ syntaks)
- Logging proxy czesto wygladaja jak decoratory

Heurystyka:
- Pytanie: 'kiedy/czy dostepu?' -> Proxy
- Pytanie: 'co zrobic przy dostepie?' -> Decorator

> 💡 W GoF (1994) Proxy i Decorator sa oddzielnymi wzorcami.
> W Python implementacja moze byc identyczna - wazna jest intencja.

In [ ]:
# Porownanie: Proxy vs Decorator na ServiceCall

class Service:
    def process(self, data: str) -> str:
        return f'processed:{data}'

# PROXY: kontroluje dostep - nie udostepni jesli brak uprawnien
class AccessProxy:
    def __init__(self, service: Service, authorized: bool):
        self._service = service
        self._authorized = authorized
    def process(self, data: str) -> str:
        if not self._authorized:
            raise PermissionError('Not authorized')
        return self._service.process(data)  # deleguje bez zmiany

# PROXY: lazy - tworzy service dopiero gdy potrzebny
class LazyProxy:
    def __init__(self):
        self._service: Service | None = None
    def process(self, data: str) -> str:
        if self._service is None:
            print('Lazy: creating Service...')
            self._service = Service()
        return self._service.process(data)

# DECORATOR: dodaje zachowanie (logowanie)
class LoggingDecorator:
    def __init__(self, service: Service):
        self._service = service
    def process(self, data: str) -> str:
        print(f'Before: processing {data!r}')
        result = self._service.process(data)
        print(f'After: {result!r}')
        return result

# DECORATOR: zmienia wynik
class UpperDecorator:
    def __init__(self, service: Service):
        self._service = service
    def process(self, data: str) -> str:
        return self._service.process(data).upper()  # modyfikuje wynik

print('--- Access Proxy ---')
AccessProxy(Service(), authorized=True).process('hello')
try:
    AccessProxy(Service(), authorized=False).process('hello')
except PermissionError as e:
    print(f'Denied: {e}')

print('--- Lazy Proxy ---')
proxy = LazyProxy()
proxy.process('test')
proxy.process('test2')

print('--- Decorator ---')
LoggingDecorator(Service()).process('world')
print(UpperDecorator(Service()).process('world'))

---

### 🐍 Cwiczenia - Proxy vs Decorator

1. Dla `ImageService.resize(img, w, h)` napisz osobno Proxy
   (lazy loading) i Decorator (logging). Porownaj implementacje.
2. Napisz `CompositeProxy` laczacy lazy loading + caching + logging
   przez lancuch obiektow (nie przez `__getattr__`).
3. *(Trudniejsze)* Sprawdz `functools.lru_cache` - czy jest Proxy
   czy Decorator? Uzasadnij odpowiedz testami.

In [ ]:
# Cwiczenie 1: Proxy vs Decorator dla ImageService
class ImageService:
    def resize(self, img: str, w: int, h: int) -> str:
        print(f'Resizing {img} to {w}x{h}')
        return f'{img}_{w}x{h}'

# Proxy: lazy loading
class LazyImageProxy:
    def __init__(self):
        self._service: ImageService | None = None
    def resize(self, img: str, w: int, h: int) -> str:
        if self._service is None:
            print('[Proxy] Creating ImageService...')
            self._service = ImageService()
        return self._service.resize(img, w, h)

# Decorator: logowanie
class LoggingImageDecorator:
    def __init__(self, service: ImageService): self._s = service
    def resize(self, img: str, w: int, h: int) -> str:
        print(f'[Log] resize({img}, {w}, {h})')
        result = self._s.resize(img, w, h)
        print(f'[Log] result: {result}')
        return result

print('Lazy Proxy:')
lazy = LazyImageProxy()
lazy.resize('photo.jpg', 800, 600)
lazy.resize('photo.jpg', 400, 300)  # service juz istnieje

print('Logging Decorator:')
log_dec = LoggingImageDecorator(ImageService())
log_dec.resize('photo.jpg', 800, 600)

In [ ]:
# Cwiczenie 2: CompositeProxy przez lancuch
class DataService:
    def fetch(self, query: str) -> list:
        print(f'[DB] Fetching: {query}')
        return [{'query': query, 'result': [1, 2, 3]}]

class CachingDataProxy:
    def __init__(self, service):
        self._service = service
        self._cache = {}
    def fetch(self, query: str) -> list:
        if query not in self._cache:
            self._cache[query] = self._service.fetch(query)
        else:
            print(f'[Cache] Hit: {query}')
        return self._cache[query]

class LoggingDataProxy:
    def __init__(self, service):
        self._service = service
    def fetch(self, query: str) -> list:
        print(f'[Log] fetch({query!r})')
        result = self._service.fetch(query)
        print(f'[Log] got {len(result)} results')
        return result

# Lancuch: LoggingProxy -> CachingProxy -> DataService
service = LoggingDataProxy(CachingDataProxy(DataService()))
service.fetch('SELECT * FROM products')
print('---')
service.fetch('SELECT * FROM products')  # cache hit

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: lru_cache - Proxy czy Decorator?
import functools

@functools.lru_cache(maxsize=128)
def expensive(n: int) -> int:
    print(f'Computing {n}...')
    return n * n

expensive(5)
expensive(5)  # z cache - nie ma 'Computing'
expensive(6)

# lru_cache jest OBYDWOMA:
# Jako Decorator: uzywamy @ syntaks, otacza funkcje
# Jako Proxy: kontroluje dostep (czy wywolac oryginal czy cache)
print('Nazwa funkcji:', expensive.__name__)   # zachowane
print('Cache info:', expensive.cache_info())  # hits, misses

# Wnioski:
conclusions = [
    'Decorator: @ syntaks, wraps zachowuje __name__/__doc__',
    'Proxy: kontroluje KIEDY wywolac oryginalna funkcje (cache check)',
    'Zalezy od perspektywy - implementacja ta sama, intencja inna',
]
for c in conclusions:
    print(f'- {c}')